# SMS Spam Classification: Accuracy-Focused SVM

This version focuses on improving test-set accuracy. It preserves useful SMS signals such as numbers, punctuation, capitalization, URLs, and repeated characters instead of aggressively deleting them. It combines word-level and character-level TF-IDF features and tunes a Linear SVM using stratified cross-validation.

**Important:** Actual accuracy is only known after running the notebook on the dataset. Do not report a score before running it.

In [ ]:
!pip -q install scikit-learn pandas matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42

## 1. Load the dataset

Upload the Kaggle `spam.csv` file if it is not already in the notebook environment. The loader accepts either `spam.csv` or `SMS.csv` and uses the standard UCI/Kaggle columns (`v1`, `v2`).

In [ ]:
# Google Colab upload is optional; skip if the CSV is already present.
if not (os.path.exists("spam.csv") or os.path.exists("SMS.csv")):
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        print("Place spam.csv or SMS.csv in the current working directory.")

filename = "spam.csv" if os.path.exists("spam.csv") else "SMS.csv"
df = pd.read_csv(filename, encoding="latin-1")

# The common Kaggle dataset uses v1=label, v2=message.
if {"v1", "v2"}.issubset(df.columns):
    df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})
elif {"label", "message"}.issubset(df.columns):
    df = df[["label", "message"]]
else:
    raise ValueError("Expected columns v1/v2 or label/message. Check the CSV format.")

df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df = df[df["label"].isin(["ham", "spam"])]
df["message"] = df["message"].astype(str)

# Remove exact duplicate rows so repeated copies do not inflate evaluation.
df = df.drop_duplicates(subset=["label", "message"]).reset_index(drop=True)
print("Dataset shape:", df.shape)
print(df["label"].value_counts())
df.head()

## 2. Train/test split

The test set is held out and is not used for tuning. Stratification keeps the ham/spam proportions similar in both sets.

In [ ]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training messages: {len(X_train)}")
print(f"Test messages: {len(X_test)}")

## 3. Word + character TF-IDF with Linear SVM

Word features learn informative words and short phrases. Character n-grams can capture spelling variations, unusual formatting, and patterns in promotional or suspicious messages. We use the original text rather than removing digits and punctuation, since those can be useful spam clues.

In [ ]:
features = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="word",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=100000
    )),
    ("char_tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=150000
    ))
])

pipeline = Pipeline([
    ("features", features),
    ("svm", LinearSVC(max_iter=10000))
])

param_grid = {
    "svm__C": [0.1, 0.3, 1, 2, 5, 10],
    "svm__class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best 5-fold CV accuracy: {search.best_score_:.4%}")

## 4. Final test-set results

These metrics are calculated on the held-out test set, which was not used to choose hyperparameters.

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_accuracy:.4%}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted ham", "Predicted spam"],
    yticklabels=["Actual ham", "Actual spam"]
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 5. Optional: inspect mistakes

This helps identify the messages the model still gets wrong. It does not change the test score.

In [ ]:
results = pd.DataFrame({
    "message": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "predicted": pd.Series(y_pred)
})
mistakes = results[results["actual"] != results["predicted"]]
print("Misclassified messages:", len(mistakes))
display(mistakes.head(30))

## 6. Predict new messages

The trained pipeline accepts raw SMS text directly; do not run a separate cleaning function.

In [ ]:
def predict_sms(message):
    prediction = best_model.predict([str(message)])[0]
    return "Spam" if prediction == "spam" else "Ham"

print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))